In [ ]:
import pandas as pd
import json
import unicodedata
from pathlib import Path

hotels_path = Path("hotels/hotels.csv")
if not hotels_path.exists():
    hotels_path = Path("../hotels/hotels.csv")

if not hotels_path.exists():
    raise FileNotFoundError(
        f"Could not find hotels.csv. Checked: {hotels_path.resolve()}"
    )

hotels = pd.read_csv(hotels_path, encoding="utf-8")
with open("cities.json", "r", encoding="utf-8") as file:
    cities = json.load(file)

cities_df = pd.DataFrame(cities)

def normalize_city(city):
    if pd.isna(city):
        return ""
  
    city = str(city).lower().strip()
    city = unicodedata.normalize("NFKD", city)
    city = "".join(c for c in city if not unicodedata.combining(c))

    return city

hotels["city_normalized"] = hotels["city"].apply(normalize_city)
cities_df["city_normalized"] = cities_df["name"].apply(normalize_city)

city_map = dict(zip(cities_df["city_normalized"], cities_df["id"]))

hotels["city_id"] = hotels["city_normalized"].map(city_map)

matched = hotels[hotels["city_id"].notna()].copy()
unmatched = hotels[hotels["city_id"].isna()].copy()

print("========== CITY MATCHING ==========")
print("Total hotel records:", len(hotels))
print("Matched records:", len(matched))
print("Unmatched records:", len(unmatched))

print("\nTotal hotel cities:", hotels["city_normalized"].nunique())
print("Matched cities:", matched["city_normalized"].nunique())
print("Unmatched cities:", unmatched["city_normalized"].nunique())

print("\n========== UNMATCHED CITIES ==========")
print(unmatched["city"].value_counts().head(30))

matched.to_csv("matched_hotels.csv", index=False)

print("\nmatched_hotels.csv created")

In [ ]:
print("========== UNMATCHED CITY CHECK ==========")

matched_names = set(cities_df["city_normalized"])

unmatched_cities = sorted(
    set(hotels["city_normalized"]) - matched_names
)

print("Unmatched unique cities:", len(unmatched_cities))
print()

for city in unmatched_cities:
    print(city)

In [ ]:
import pandas as pd
import json
import unicodedata
from rapidfuzz import process, fuzz

hotels = pd.read_csv("hotels/hotels.csv")

with open("cities.json", "r", encoding="utf-8") as file:
    cities = json.load(file)

cities_df = pd.DataFrame(cities)

def normalize_city(city):
    if pd.isna(city):
        return ""

    city = str(city).lower().strip()
    city = unicodedata.normalize("NFKD", city)
    city = "".join(c for c in city if not unicodedata.combining(c))

    return city

hotels["city_normalized"] = hotels["city"].apply(normalize_city)
cities_df["city_normalized"] = cities_df["name"].apply(normalize_city)

city_map = dict(zip(cities_df["city_normalized"], cities_df["id"]))

hotels["city_id"] = hotels["city_normalized"].map(city_map)

unmatched = hotels[hotels["city_id"].isna()].copy()

city_names = cities_df["city_normalized"].tolist()

print("========== FUZZY CITY MATCHING ==========")

for city in sorted(unmatched["city_normalized"].unique()):
    result = process.extractOne(
        city,
        city_names,
        scorer=fuzz.ratio
    )

    print(city, "->", result)

In [ ]:
city_aliases = {
    "bangalore": "bengaluru",
    "cochin": "kochi",
    "calcutta": "kolkata",
    "trivandrum": "thiruvananthapuram",
    "alleppey": "alappuzha",
    "ooty": "udhagamandalam",
    "baroda": "vadodara",
    "pondicherry": "puducherry",
    "benaras": "varanasi",
    "benares": "varanasi",
    "ahmadnagar": "ahmednagar"
}

In [ ]:
hotels["canonical_city"] = hotels["city_normalized"].replace(city_aliases)

hotels["city_id"] = hotels["canonical_city"].map(
    dict(zip(cities_df["city_normalized"], cities_df["id"]))
)

In [ ]:
print("Total hotels:", len(hotels))
print("Matched:", hotels["city_id"].notna().sum())
print("Unmatched:", hotels["city_id"].isna().sum())

In [ ]:
alias_rows = hotels[hotels["city_normalized"].isin(city_aliases)]

print(alias_rows[
    ["city", "canonical_city", "state", "city_id"]
].drop_duplicates().to_string(index=False))

In [ ]:
print("Exact matches:", exact_match.sum())
print("After verified aliases:", hotels["city_id"].notna().sum())
print("Still unmatched:", hotels["city_id"].isna().sum())

In [ ]:
unmatched = hotels[hotels["city_id"].isna()]

print("Unmatched unique cities:", unmatched["canonical_city"].nunique())
print("\nTop unmatched cities:")
print(unmatched["canonical_city"].value_counts().head(30))

In [ ]:
cleaned_hotels = hotels[
    hotels["city_id"].notna() &
    hotels["property_name"].notna() &
    hotels["property_id"].notna()
].copy()

print("Rows:", len(cleaned_hotels))

In [ ]:
cleaned_hotels["site_review_count"] = pd.to_numeric(
    cleaned_hotels["site_review_count"],
    errors="coerce"
)

In [ ]:
print(cleaned_hotels["site_review_count"].dtype)

In [ ]:
cleaned_hotels["hotel_star_rating"] = (
    cleaned_hotels["hotel_star_rating"]
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

In [ ]:
city_names = set(cities_df["city_normalized"])

for old, new in city_aliases.items():
    if new in city_names:
        print(old, "->", new)
    else:
        print("NOT FOUND:", old, "->", new)

In [ ]:
hotels[["city", "state"]].head(20)

In [ ]:
cities_df.head(20)

In [ ]:
print("Hotels with city:", hotels["city"].notna().sum())
print("Hotels with state:", hotels["state"].notna().sum())
print("Cities with state:", cities_df["state"].notna().sum())

In [ ]:
exact_match = hotels["city_normalized"].isin(set(cities_df["city_normalized"]))

print("Exact matched hotels:", exact_match.sum())
print("Exact unmatched hotels:", (~exact_match).sum())

In [ ]:
cleaned_hotels["hotel_star_rating"] = (
    cleaned_hotels["hotel_star_rating"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)
print(cleaned_hotels["hotel_star_rating"].value_counts(dropna=False).sort_index())

In [ ]:
stay_data = cleaned_hotels[
    [
        "property_id",
        "city_id",
        "property_name",
        "property_type",
        "hotel_star_rating",
        "site_review_rating",
        "site_review_count",
        "hotel_facilities",
        "latitude",
        "longitude"
    ]
].copy()

In [ ]:
stay_data["stay_id"] = stay_data["property_id"].astype(int).astype(str)

In [ ]:
stay_data = stay_data[
    [
        "stay_id",
        "city_id",
        "property_name",
        "property_type",
        "hotel_star_rating",
        "site_review_rating",
        "site_review_count",
        "hotel_facilities",
        "latitude",
        "longitude"
    ]
]

In [ ]:
print("STAY rows:", len(stay_data))
print("STAY columns:", len(stay_data.columns))
print(stay_data.head())

In [ ]:
city_data = cities_df[
    ["id", "name", "state"]
].copy()
city_data.columns = ["city_id", "city_name", "state"]

In [ ]:
print("CITY rows:", len(city_data))
print("CITY columns:", len(city_data.columns))
print(city_data.head())

In [ ]:
city_data.to_csv("city_data.csv", index=False)
print("city_data.csv created")

In [ ]:
print("HOTEL DATASET")
print("Rows:", len(hotels))
print("Columns:", len(hotels.columns))

print("\nMissing property name:", hotels["property_name"].isna().sum())
print("Missing property ID:", hotels["property_id"].isna().sum())
print("Missing city:", hotels["city"].isna().sum())
print("\nCITY MAPPING")
print("Exact + alias mapped:", hotels["city_id"].notna().sum())
print("Unmapped:", hotels["city_id"].isna().sum())

In [ ]:
print("Total:", len(hotels))

print("\nBoth property name and ID present:",
      hotels["property_name"].notna().sum())

print("City mapped:",
      hotels["city_id"].notna().sum())

print("Property + city mapped:",
      (hotels["property_name"].notna() &
       hotels["property_id"].notna() &
       hotels["city_id"].notna()).sum())

In [ ]:
print("\nUnmapped hotels with property information:")

unmapped = hotels[
    hotels["city_id"].isna() &
    hotels["property_name"].notna() &
    hotels["property_id"].notna()
]

print("Count:", len(unmapped))
print(
    unmapped[
        ["property_name", "city", "state", "latitude", "longitude"]
    ].head(20).to_string(index=False)
)

In [ ]:
stay_data.to_csv("stay_data.csv", index=False)
print("stay_data.csv created")

